# Self-Consistency & Tree of Thoughts

Both techniques have the same core idea:

> **Don't just ask the AI once. Explore multiple answers, then pick the best one.**

---

### Self-Consistency = Ask the same question 3-5 times, majority vote wins
Like asking 5 friends the same question and going with what most of them say.

### Tree of Thoughts = At each step, explore 3 options, keep the best one
Like a chess player thinking — 'if I go here... or here... or here...' before moving.

---
## Setup

In [ ]:
import boto3
import json
import re
from collections import Counter

MODEL_NAME = "us.amazon.nova-lite-v1:0"
# MODEL_NAME = "anthropic.claude-3-haiku-20240307-v1:0"
AWS_REGION = "us-east-1"

bedrock = boto3.client('bedrock-runtime', region_name=AWS_REGION)


def get_completion(prompt, system='', temperature=0.0, model_name=MODEL_NAME):
    if model_name.startswith("anthropic.claude"):
        body_dict = {
            "anthropic_version": "bedrock-2023-05-31",
            "max_tokens": 1024,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": temperature,
            "top_p": 1,
            "system": system
        }
        response = bedrock.invoke_model(body=json.dumps(body_dict), modelId=model_name)
        response_body = json.loads(response.get("body").read())
        return response_body.get("content")[0].get("text")

    elif "amazon.nova" in model_name:
        body_dict = {
            "messages": [{"role": "user", "content": [{"text": prompt}]}],
            "inferenceConfig": {
                "max_new_tokens": 1024,
                "temperature": temperature,
                "top_p": 1,
            }
        }
        if system:
            body_dict["system"] = [{"text": system}]
        response = bedrock.invoke_model(body=json.dumps(body_dict), modelId=model_name)
        response_body = json.loads(response.get("body").read())
        return response_body.get("output").get("message").get("content")[0].get("text")

    else:
        raise ValueError(f"Unsupported model: {model_name}")


print("Ready!")

---
# Part 1: Self-Consistency

**The idea:** Ask the same question 5 times. Each time the model reasons slightly differently.
Count the answers. The most common one wins.

**Why temperature=0.7?**
At temperature=0 the model gives the exact same answer every time — useless for voting.
At temperature=0.7 each run is slightly different, like asking 5 different people.

In [ ]:
question = """A shop sells apples for $1.50 and oranges for $2.00.
John buys 4 apples and 3 oranges and pays with a $20 note.
How much change does he get? Write Final Answer: $<number> at the end."""

print("Running the same question 5 times...\n")

answers = []

for i in range(1, 6):
    response = get_completion(question, temperature=0.7)
    # Pull out the final number
    match = re.search(r'Final Answer:\s*\$?([\d\.]+)', response)
    answer = match.group(1) if match else "?"
    answers.append(answer)
    print(f"Run {i} → ${answer}")

# Count votes
vote_counts = Counter(answers)
winner = vote_counts.most_common(1)[0][0]

print("\n--- Vote tally ---")
for ans, count in vote_counts.most_common():
    print(f"  ${ans}  {'█' * count}  ({count}/5 votes)")

print(f"\n✅ Self-Consistency Answer: ${winner}")

---
# Part 2: Tree of Thoughts

**The idea:** At each step, generate 3 possible next thoughts.
Score each one. Keep the best. Go one step deeper. Repeat.

```
Step 1:  Thought A (7/10)  Thought B (4/10)  Thought C (8/10)  → keep C
Step 2:  Thought C1(6/10) Thought C2(9/10)  Thought C3(5/10)  → keep C2
Step 3:  Final answer based on best path: C → C2
```

**Best for:** planning problems, open-ended questions, anything needing exploration.

### Step A — Generate 3 different thoughts

In [ ]:
problem = "How can a small coffee shop compete against a big chain that just opened nearby?"

# Step 1: Ask the model for 3 different approaches
generate_prompt = f"""Problem: {problem}

Give 3 completely different approaches to solve this.
Number them 1, 2, 3. One sentence each."""

response = get_completion(generate_prompt, temperature=0.7)
print("3 possible thoughts generated:")
print(response)

# Split into a list
thoughts = [line.strip() for line in response.strip().split('\n') if re.match(r'^[123]\.', line.strip())]
print(f"\nExtracted {len(thoughts)} thoughts")

### Step B — Score each thought (1-10)

In [ ]:
print("Scoring each thought...\n")

scored_thoughts = []

for thought in thoughts:
    score_prompt = f"""Problem: {problem}
Approach: {thought}

Rate how good this approach is from 1 to 10.
Reply with just the number."""

    score_response = get_completion(score_prompt, temperature=0.0)
    match = re.search(r'\b([1-9]|10)\b', score_response)
    score = int(match.group(1)) if match else 5
    scored_thoughts.append((score, thought))
    print(f"Score {score}/10 → {thought[:80]}")

# Pick the best one
scored_thoughts.sort(reverse=True)
best_score, best_thought = scored_thoughts[0]
print(f"\n✅ Best thought (score {best_score}/10):")
print(best_thought)

### Step C — Go one level deeper on the best thought

In [ ]:
# Now branch again from the best thought
deeper_prompt = f"""Problem: {problem}
We decided to: {best_thought}

Now give 3 specific actions to implement this.
Number them 1, 2, 3. One sentence each."""

deeper_response = get_completion(deeper_prompt, temperature=0.7)
print("3 ways to implement the best thought:")
print(deeper_response)

deeper_thoughts = [line.strip() for line in deeper_response.strip().split('\n') if re.match(r'^[123]\.', line.strip())]

# Score again
print("\nScoring...")
scored_deeper = []
for thought in deeper_thoughts:
    score_prompt = f"""Problem: {problem}\nAction: {thought}\nRate 1-10. Reply with just the number."""
    score_response = get_completion(score_prompt, temperature=0.0)
    match = re.search(r'\b([1-9]|10)\b', score_response)
    score = int(match.group(1)) if match else 5
    scored_deeper.append((score, thought))
    print(f"  Score {score}/10 → {thought[:80]}")

scored_deeper.sort(reverse=True)
best_action = scored_deeper[0][1]
print(f"\n✅ Best action: {best_action}")

### Step D — Final answer using the best path

In [ ]:
final_prompt = f"""Problem: {problem}

Best strategy chosen: {best_thought}
Best action to take: {best_action}

Write a short, clear final recommendation (3-4 sentences)."""

final_answer = get_completion(final_prompt, temperature=0.0)
print("FINAL ANSWER (Tree of Thoughts):")
print("=" * 50)
print(final_answer)

---
## Summary

| | Self-Consistency | Tree of Thoughts |
|---|---|---|
| What it does | Ask same question N times, majority wins | Branch → score → keep best → go deeper |
| Best for | Math, logic, one right answer | Planning, strategy, open-ended |
| Temperature | 0.7 (needs variety) | 0.7 for branching, 0.0 for scoring |
| Simple analogy | Ask 5 friends, go with majority | Chess player thinking ahead |